# Airline Operations & Disruption Intelligence
## 02 — Data Quality Assessment

### Objective

The objective of this notebook is to assess the quality and reliability
of the combined BTS flight dataset before performing data cleaning.

We will identify:

- Missing values
- Duplicate records
- Invalid dates
- Unexpected categorical values
- Invalid numerical values
- Impossible delay values
- Flight status inconsistencies
- Potential data quality issues

### Important Rule

This stage identifies problems.

We will NOT automatically delete or modify records.

Cleaning decisions will be made in the next stage.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path(r"D:\Data Analyst\EXCEL\api\processed\flights_2026_q1.csv")

flights = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Shape:", flights.shape)

Dataset loaded successfully.
Shape: (1847242, 43)


## 2. Basic Structure Check

Before checking individual quality issues, we verify that the dataset
still contains the expected number of rows and columns.

In [2]:
print("Rows:", flights.shape[0])
print("Columns:", flights.shape[1])

Rows: 1847242
Columns: 43


In [3]:
print(flights.columns.tolist())

['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'MKT_UNIQUE_CARRIER', 'BRANDED_CODE_SHARE', 'MKT_CARRIER_AIRLINE_ID', 'MKT_CARRIER', 'MKT_CARRIER_FL_NUM', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_NM', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'TAXI_OUT', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']


## 3. Missing Value Assessment

Missing values do not automatically mean that the data is incorrect.

For example, a flight that was cancelled may not have an actual departure
time or arrival time.

Therefore, we first measure missing values before deciding how to handle them.

In [4]:
missing_count = flights.isnull().sum()
missing_count.sort_values(ascending=False)

CANCELLATION_CODE         1784931
LATE_AIRCRAFT_DELAY       1462816
CARRIER_DELAY             1462816
SECURITY_DELAY            1462816
NAS_DELAY                 1462816
WEATHER_DELAY             1462816
AIR_TIME                    67057
ACTUAL_ELAPSED_TIME         67057
ARR_DELAY_NEW               67048
ARR_DELAY                   67048
ARR_DEL15                   67048
ARR_TIME                    62950
TAXI_IN                     62950
TAXI_OUT                    61991
DEP_DELAY                   60974
DEP_DEL15                   60974
DEP_DELAY_NEW               60974
DEP_TIME                    60784
CRS_ELAPSED_TIME                1
MONTH                           0
QUARTER                         0
YEAR                            0
BRANDED_CODE_SHARE              0
MKT_UNIQUE_CARRIER              0
FL_DATE                         0
DAY_OF_WEEK                     0
DAY_OF_MONTH                    0
MKT_CARRIER_FL_NUM              0
MKT_CARRIER                     0
MKT_CARRIER_AI

In [5]:
missing_percentage = (
    flights.isnull().mean() * 100
).sort_values(ascending=False)

missing_percentage[missing_percentage > 0]

CANCELLATION_CODE      96.626809
LATE_AIRCRAFT_DELAY    79.189191
CARRIER_DELAY          79.189191
SECURITY_DELAY         79.189191
NAS_DELAY              79.189191
WEATHER_DELAY          79.189191
AIR_TIME                3.630115
ACTUAL_ELAPSED_TIME     3.630115
ARR_DELAY_NEW           3.629627
ARR_DELAY               3.629627
ARR_DEL15               3.629627
ARR_TIME                3.407783
TAXI_IN                 3.407783
TAXI_OUT                3.355868
DEP_DELAY               3.300813
DEP_DEL15               3.300813
DEP_DELAY_NEW           3.300813
DEP_TIME                3.290527
CRS_ELAPSED_TIME        0.000054
dtype: float64

In [6]:
quality_summary = pd.DataFrame({
    "missing_count": flights.isnull().sum(),
    "missing_percentage": flights.isnull().mean() * 100,
    "data_type": flights.dtypes.astype(str)
})

quality_summary = quality_summary.sort_values(
    "missing_percentage",
    ascending=False
)

quality_summary

,missing_count,missing_percentage,data_type
CANCELLATION_CODE,1784931,96.626809,object
LATE_AIRCRAFT_DELAY,1462816,79.189191,float64
CARRIER_DELAY,1462816,79.189191,float64
SECURITY_DELAY,1462816,79.189191,float64
NAS_DELAY,1462816,79.189191,float64
WEATHER_DELAY,1462816,79.189191,float64
AIR_TIME,67057,3.630115,float64
ACTUAL_ELAPSED_TIME,67057,3.630115,float64
ARR_DELAY_NEW,67048,3.629627,float64
ARR_DELAY,67048,3.629627,float64


## 4. Duplicate Record Assessment

Duplicate records can cause flight counts, delay rates, and other metrics
to be overstated.

We first identify the number of completely duplicated rows.

In [7]:
duplicate_count = flights.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 0


In [8]:
duplicate_percentage = (
    duplicate_count / len(flights)
) * 100

print(
    f"Duplicate percentage: {duplicate_percentage:.2f}%"
)

Duplicate percentage: 0.00%


## 5. Flight Date Quality

The flight date should contain valid dates within the expected
January–March 2026 analysis period.

We will check for invalid or missing dates.

In [9]:
flights["FL_DATE"] = pd.to_datetime(
    flights["FL_DATE"],
    errors="coerce"
)

print("Missing/invalid FL_DATE:", flights["FL_DATE"].isna().sum())

Missing/invalid FL_DATE: 0


In [10]:
print("Minimum date:", flights["FL_DATE"].min())
print("Maximum date:", flights["FL_DATE"].max())

Minimum date: 2026-01-01 00:00:00
Maximum date: 2026-03-31 00:00:00


## 6. Flight Status Quality

We inspect the values of cancellation and diversion indicators.

These variables should represent whether a flight was cancelled or diverted.

In [11]:
print("CANCELLED:")
print(flights["CANCELLED"].value_counts(dropna=False))

print("\nDIVERTED:")
print(flights["DIVERTED"].value_counts(dropna=False))

CANCELLED:
CANCELLED
0.0    1784931
1.0      62311
Name: count, dtype: int64

DIVERTED:
DIVERTED
0.0    1842505
1.0       4737
Name: count, dtype: int64


## 7. Delay Indicator Quality

BTS provides indicators such as DEP_DEL15 and ARR_DEL15.

These indicate whether the departure or arrival delay was at least
15 minutes.

We check whether the values are valid binary indicators.

In [12]:
print("DEP_DEL15:")
print(flights["DEP_DEL15"].value_counts(dropna=False))

print("\nARR_DEL15:")
print(flights["ARR_DEL15"].value_counts(dropna=False))

DEP_DEL15:
DEP_DEL15
0.0    1396419
1.0     389849
NaN      60974
Name: count, dtype: int64

ARR_DEL15:
ARR_DEL15
0.0    1395768
1.0     384426
NaN      67048
Name: count, dtype: int64


## 8. Delay Value Assessment

Departure and arrival delays are measured in minutes.

Negative values can be valid because a flight can arrive or depart
earlier than scheduled.

Therefore, negative delay values should NOT automatically be treated
as errors.

In [13]:
delay_columns = [
    "DEP_DELAY",
    "DEP_DELAY_NEW",
    "ARR_DELAY",
    "ARR_DELAY_NEW"
]

flights[delay_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
DEP_DELAY,1786268.0,13.938069,61.113006,-66.0,-7.0,-2.0,10.0,3339.0
DEP_DELAY_NEW,1786268.0,17.586258,59.914301,0.0,0.0,0.0,10.0,3339.0
ARR_DELAY,1780194.0,7.249089,63.468663,-90.0,-18.0,-7.0,10.0,3339.0
ARR_DELAY_NEW,1780194.0,17.345602,59.634120,0.0,0.0,0.0,10.0,3339.0


In [14]:
for column in ["DEP_DELAY", "ARR_DELAY"]:
    print(f"\n{column}")
    print("Minimum:", flights[column].min())
    print("Maximum:", flights[column].max())


DEP_DELAY
Minimum: -66.0
Maximum: 3339.0

ARR_DELAY
Minimum: -90.0
Maximum: 3339.0


## 9. Distance Quality

Flight distance should not be negative.

We check for zero or negative distances.

In [15]:
invalid_distance = flights[
    flights["DISTANCE"] <= 0
]

print(
    "Records with zero or negative distance:",
    len(invalid_distance)
)

Records with zero or negative distance: 0


In [16]:
time_columns = [
    "TAXI_OUT",
    "TAXI_IN",
    "CRS_ELAPSED_TIME",
    "ACTUAL_ELAPSED_TIME",
    "AIR_TIME"
]

flights[time_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
TAXI_OUT,1785251.0,19.200850,11.130109,1.0,13.0,16.0,22.0,224.0
TAXI_IN,1784292.0,8.786492,7.780903,1.0,5.0,7.0,10.0,341.0
CRS_ELAPSED_TIME,1847241.0,148.070241,72.255269,-85.0,95.0,131.0,180.0,1202.0
ACTUAL_ELAPSED_TIME,1780185.0,141.567923,72.431443,14.0,89.0,125.0,173.0,763.0
AIR_TIME,1780185.0,113.598469,70.744396,6.0,62.0,97.0,144.0,719.0


## 10. Airport Code Quality

Origin and destination airport codes are important because they will later
be joined with the OpenFlights airport reference dataset.

At this stage, we check for missing airport codes and inspect the number
of unique values.

In [17]:
print("Missing ORIGIN:", flights["ORIGIN"].isna().sum())
print("Missing DEST:", flights["DEST"].isna().sum())

print("Unique ORIGIN:", flights["ORIGIN"].nunique())
print("Unique DEST:", flights["DEST"].nunique())

Missing ORIGIN: 0
Missing DEST: 0
Unique ORIGIN: 362
Unique DEST: 362


In [18]:
print(
    flights["MKT_UNIQUE_CARRIER"]
    .value_counts(dropna=False)
)

MKT_UNIQUE_CARRIER
AA    483902
DL    386832
UA    358108
WN    327083
AS    116489
B6     57320
F9     50557
NK     36379
G4     30572
Name: count, dtype: int64


## 11. Delay Cause Assessment

BTS provides several delay-cause variables.

These variables may be missing when a flight does not have the
corresponding type of delay.

Therefore, missing values here should be interpreted carefully rather
than automatically treated as bad data.

In [19]:
delay_cause_columns = [
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

flights[delay_cause_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
CARRIER_DELAY,384426.0,26.771269,82.074800,0.0,0.0,3.0,23.0,3339.0
WEATHER_DELAY,384426.0,5.317260,41.409899,0.0,0.0,0.0,0.0,1958.0
NAS_DELAY,384426.0,14.499233,34.940501,0.0,0.0,0.0,18.0,1560.0
SECURITY_DELAY,384426.0,0.130366,4.280014,0.0,0.0,0.0,0.0,1600.0
LATE_AIRCRAFT_DELAY,384426.0,29.269703,64.896116,0.0,0.0,0.0,33.0,2338.0


## 12. Cancellation Code Assessment

Cancellation codes should be interpreted together with the CANCELLED
indicator.

We first inspect the available values.

In [20]:
flights["CANCELLATION_CODE"].value_counts(
    dropna=False
)

CANCELLATION_CODE
NaN    1784931
B        47348
A        10042
C         4852
D           69
Name: count, dtype: int64

## 13. Data Quality Summary

The following checks were performed:

1. Dataset structure
2. Missing values
3. Duplicate records
4. Flight date validity
5. Calendar variable validity
6. Cancellation and diversion indicators
7. Delay indicators
8. Delay values
9. Distance values
10. Operational duration values
11. Airport codes
12. Airline codes
13. Delay causes
14. Cancellation codes

No records have been removed or modified during this assessment.